# Comparación de algoritmos de aprendizaje supervisado: KNN, SVM y MLP

**Problema:** clasificación de tumores de mama en **benignos** y **malignos**
**Dataset:** Wisconsin Diagnostic Breast Cancer (WDBC) — archivo `wdbc.data` (descripción en `wdbc.names`)

| Integrante | Rol / sección |
|---|---|
| _Nombre (líder)_ | Base del proyecto (carga, exploración, preprocesamiento), MLP e integración |
| _Nombre_ | KNN |
| Owen | SVM |
| _Nombre_ | Comparación de modelos |
| Frida Natalia Dzul Chi | Conclusiones |

### Índice
0. Introducción
1. Carga y exploración del dataset
2. Preprocesamiento
3. K-Nearest Neighbors (KNN)
4. Support Vector Machine (SVM)
5. Red neuronal artificial (MLP)
6. Comparación de modelos
7. Conclusiones

> En Colab puedes navegar entre secciones con el panel **Índice / Table of contents** (icono ☰ de la barra lateral izquierda).
> Para reproducir todos los resultados: **Entorno de ejecución → Reiniciar y ejecutar todo**.

## 0. Introducción

El objetivo de esta actividad es **implementar y comparar tres algoritmos de aprendizaje supervisado**
—K-Nearest Neighbors (KNN), Support Vector Machine (SVM) y una red neuronal artificial tipo
Perceptrón Multicapa (MLP)— sobre **el mismo conjunto de datos y la misma división
entrenamiento/prueba**, para determinar cuál ofrece mejores resultados en un problema de
**clasificación binaria**.

**Sobre el dataset.** Cada registro describe las características de los núcleos celulares
observados en una imagen digitalizada de una punción con aguja fina (PAAF) de una masa mamaria.
Se calculan 10 rasgos (radio, textura, perímetro, área, suavidad, compacidad, concavidad,
puntos cóncavos, simetría y dimensión fractal) y, para cada uno, tres estadísticos:
la **media**, el **error estándar** y el **peor valor** (media de los tres valores más grandes),
lo que da **30 variables numéricas**. La variable a predecir es `diagnosis`: **M** (maligno) o **B** (benigno).

**Metodología común para los tres modelos**
1. Cargar y explorar el dataset.
2. Separar características (X) y variable objetivo (y) y codificar: maligno = 1, benigno = 0.
3. Dividir en entrenamiento (80 %) y prueba (20 %) de forma estratificada con `random_state=42`.
4. Estandarizar las características (ajustando el escalador **solo** con el entrenamiento).
5. Entrenar cada modelo y evaluarlo con los **mismos datos de prueba**: accuracy, precision,
   recall, F1-score y matriz de confusión.

> **Clase positiva = Maligno (1).** Precision, recall y F1 se calculan respecto a los tumores malignos,
> que es la clase que más importa detectar.

## 1. Carga y exploración del dataset

### 1.1 Importación de librerías
Se importan las librerías de manipulación de datos (`pandas`, `numpy`), visualización
(`matplotlib`, `seaborn`) y de *machine learning* (`scikit-learn`). Se fija una semilla global
(`RANDOM_STATE = 42`) para que cualquier persona que ejecute el notebook obtenga los mismos resultados.

In [ ]:
# ============================================================
# 1.1 Importación de librerías y configuración general
# ============================================================
import io
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
# Nota: las clases de KNN y SVM se importan en sus propias secciones.

# Semilla global para que los resultados sean reproducibles
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de las gráficas y de las tablas
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

print("Librerías cargadas correctamente.")

### 1.2 Carga del dataset
El archivo original `wdbc.data` **no trae encabezado**: los nombres de las columnas se toman de
`wdbc.names` (columna 1 = `id`, columna 2 = `diagnosis`, columnas 3–32 = las 30 características,
primero las 10 medias, luego los 10 errores estándar y después los 10 "peores" valores).

Se ofrecen tres formas de cargar el archivo (se usa la primera que esté disponible):
- **A. URL raw de GitHub** (recomendada para la entrega final: el notebook corre sin subir nada).
- **B. Archivo local** (`wdbc.data` o `data/wdbc.data`).
- **C. Subirlo manualmente a Colab** con `files.upload()`.

La función también acepta la versión CSV del dataset con encabezado (por ejemplo, la de Kaggle).

In [ ]:
# ============================================================
# 1.2 Configuración de la fuente de datos
# ============================================================
# Opción A: URL "raw" del archivo en el repositorio de GitHub del equipo.
#   Si todavía no está subido el archivo, el notebook pasa automáticamente a las opciones B y C.
URL_GITHUB = "https://raw.githubusercontent.com/imjustVEGA/ActividadIA-Comparacion_Algoritmos/main/data/wdbc.data"

# Opción B: rutas locales donde se buscará el archivo (en orden).
RUTAS_LOCALES = ["wdbc.data", "data/wdbc.data", "../data/wdbc.data", "data.csv"]

# Opción C: si no se encuentra por A ni por B y estamos en Colab, se abrirá un
#           cuadro de diálogo para subir el archivo con files.upload().

# Nombres de las columnas según wdbc.names
RASGOS = ["radius", "texture", "perimeter", "area", "smoothness",
          "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
ESTADISTICOS = ["mean", "se", "worst"]   # media, error estándar, peor valor
COLUMNAS = ["id", "diagnosis"] + [f"{r}_{e}" for e in ESTADISTICOS for r in RASGOS]
print(f"Se definieron {len(COLUMNAS)} nombres de columna (id + diagnosis + 30 características).")

In [ ]:
# ============================================================
# 1.2 Funciones de carga
# ============================================================
def leer_dataset(origen):
    # Lee el archivo y asigna nombres de columna si no trae encabezado.
    df_tmp = pd.read_csv(origen, header=None)
    primera_fila = df_tmp.iloc[0].astype(str).str.strip().str.lower().tolist()

    if "diagnosis" in primera_fila:
        # El archivo ya trae encabezado (p. ej. versión CSV de Kaggle): se vuelve a leer con él
        if hasattr(origen, "seek"):
            origen.seek(0)
        return pd.read_csv(origen)

    # Archivo original wdbc.data (sin encabezado)
    if df_tmp.shape[1] != len(COLUMNAS):
        raise ValueError(f"Se esperaban {len(COLUMNAS)} columnas y el archivo tiene {df_tmp.shape[1]}.")
    df_tmp.columns = COLUMNAS
    return df_tmp


def cargar_datos():
    # Opción A: GitHub
    if URL_GITHUB:
        try:
            datos = leer_dataset(URL_GITHUB)
            print(f"Dataset cargado desde GitHub: {URL_GITHUB}")
            return datos
        except Exception as error:
            print(f"No se pudo leer desde GitHub ({type(error).__name__}). Se intenta la carga local...")

    # Opción B: archivo local
    for ruta in RUTAS_LOCALES:
        if os.path.exists(ruta):
            print(f"Cargando el dataset desde el archivo local: {ruta}")
            return leer_dataset(ruta)

    # Opción C: subir el archivo en Colab
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("No se encontró el dataset. Define URL_GITHUB o coloca "
                                "wdbc.data junto al notebook.")
    print("Selecciona el archivo wdbc.data (o el CSV del dataset) para subirlo:")
    subidos = files.upload()
    nombre_archivo = next(iter(subidos))
    print(f"Archivo recibido: {nombre_archivo}")
    return leer_dataset(io.BytesIO(subidos[nombre_archivo]))


df_original = cargar_datos()
df = df_original.copy()   # trabajamos sobre una copia para conservar el original
print("Dataset cargado. Dimensiones (filas, columnas):", df.shape)

### 1.3 Exploración inicial
Revisamos las dimensiones, las primeras filas, los tipos de datos, los valores nulos y las
estadísticas descriptivas. Esto permite confirmar que el archivo se leyó bien y detectar
problemas antes de entrenar.

In [ ]:
# Dimensiones y primeras filas
print(f"Número de registros (filas): {df.shape[0]}")
print(f"Número de columnas         : {df.shape[1]}")
display(df.head())

In [ ]:
# Tipos de datos de cada columna
df.info()
print("\nResumen de tipos de datos:")
print(df.dtypes.value_counts())

In [ ]:
# Valores nulos y registros duplicados
nulos = df.isnull().sum()
print(f"Total de valores nulos en el dataset: {nulos.sum()}")
if nulos.sum() > 0:
    print("Columnas con valores nulos:")
    print(nulos[nulos > 0])

print(f"Registros duplicados: {df.duplicated().sum()}")

In [ ]:
# Estadísticas descriptivas de las variables numéricas.
display(df.drop(columns=["id"], errors="ignore").describe().T.round(4))

**Observación:** las variables tienen escalas muy distintas (por ejemplo, `area_mean` está en los
cientos o miles, mientras que `smoothness_mean` está alrededor de 0.1). Esto confirma que será
necesario **estandarizar** las características antes de entrenar KNN, SVM y MLP, ya que los tres
son sensibles a la escala (KNN y SVM usan distancias; la red neuronal entrena mejor con entradas
centradas y de varianza similar).

### 1.4 Variable objetivo y distribución de clases
La variable a predecir es `diagnosis`, con dos clases: **B = benigno** y **M = maligno**.
Revisamos cuántos registros hay de cada clase para saber si el problema está balanceado.

In [ ]:
# Conteo y porcentaje por clase
conteo = df["diagnosis"].value_counts()
porcentaje = df["diagnosis"].value_counts(normalize=True) * 100
resumen_clases = pd.DataFrame({"Conteo": conteo, "Porcentaje (%)": porcentaje.round(2)})
resumen_clases.index = resumen_clases.index.map({"B": "Benigno (B)", "M": "Maligno (M)"})
display(resumen_clases)

# Gráfica de barras de la distribución de clases
fig, ax = plt.subplots(figsize=(6, 4))
colores = {"Benigno (B)": "#4C9F70", "Maligno (M)": "#C8553D"}
barras = ax.bar(resumen_clases.index, resumen_clases["Conteo"],
                color=[colores[c] for c in resumen_clases.index])
for barra, (n, p) in zip(barras, resumen_clases.values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 5,
            f"{int(n)} ({p:.1f} %)", ha="center", fontsize=11)
ax.set_title("Distribución de la variable objetivo (diagnosis)")
ax.set_ylabel("Número de tumores")
ax.set_ylim(0, resumen_clases["Conteo"].max() * 1.15)
plt.tight_layout()
plt.show()

**Observación:** las clases están **moderadamente desbalanceadas** (≈ 63 % benignos y ≈ 37 % malignos).
Por eso: (1) la división entrenamiento/prueba se hará **estratificada** para conservar estas
proporciones, y (2) no basta con mirar la exactitud: también hay que revisar precision, recall y F1
de la clase maligna.

### 1.5 Distribución de algunas características por clase
Se grafican algunas de las variables "media" separadas por diagnóstico, para ver de forma intuitiva
qué tan bien distinguen los tumores benignos de los malignos.

In [ ]:
# Histogramas de 6 características "mean" separadas por diagnóstico
columnas_media = [c for c in df.columns if str(c).endswith("_mean")][:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), columnas_media):
    sns.histplot(data=df, x=col, hue="diagnosis", hue_order=["B", "M"],
                 palette={"B": "#4C9F70", "M": "#C8553D"}, kde=True,
                 element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(col)
fig.suptitle("Distribución de características por diagnóstico (B = benigno, M = maligno)", fontsize=13)
plt.tight_layout()
plt.show()

**Observación:** en variables como el radio, el perímetro, el área o la concavidad, los tumores
malignos tienden a tener valores más altos, aunque hay zonas de solapamiento entre clases. Los
modelos tendrán que aprender a separar precisamente esos casos dudosos.

### 1.6 Eliminación de columnas que no aportan
- **`id`**: es un número de identificación del paciente/muestra. No describe al tumor, por lo que no
  tiene relación con el diagnóstico. Si se dejara, el modelo podría "memorizar" identificadores en
  lugar de aprender patrones reales.
- **Columnas vacías** (por ejemplo `Unnamed: 32` en la versión CSV de Kaggle): no contienen datos.

Las 30 características numéricas y `diagnosis` se conservan.

In [ ]:
# Columnas a eliminar: el identificador y cualquier columna "Unnamed" o completamente vacía
columnas_a_eliminar = ["id"] + [c for c in df.columns
                                if str(c).startswith("Unnamed") or df[c].isnull().all()]
columnas_a_eliminar = [c for c in dict.fromkeys(columnas_a_eliminar) if c in df.columns]

df = df.drop(columns=columnas_a_eliminar)
print("Columnas eliminadas:", columnas_a_eliminar)
print("Dimensiones después de la limpieza:", df.shape)

In [ ]:
# ============================================================
# 1.7 Resumen del dataset
# ============================================================
variables_entrada = [c for c in df.columns if c != "diagnosis"]
print("RESUMEN DEL DATASET")
print("-" * 45)
print(f"Número de registros           : {df.shape[0]}")
print(f"Número de variables de entrada: {len(variables_entrada)}")
print("Variable objetivo             : diagnosis")
print("Clases a predecir             : B = Benigno, M = Maligno")
print(f"  - Benignos (B): {(df['diagnosis'] == 'B').sum()}")
print(f"  - Malignos (M): {(df['diagnosis'] == 'M').sum()}")
print(f"Valores nulos                 : {df.isnull().sum().sum()}")

## 2. Preprocesamiento

### 2.1 Separación de X (características) e y (objetivo) y codificación
- **X**: las 30 características numéricas.
- **y**: `diagnosis` codificada como **maligno = 1** y **benigno = 0**. Los modelos de scikit-learn
  trabajan con números y, al definir el maligno como clase positiva (1), las métricas precision,
  recall y F1 miden directamente qué tan bien se detectan los tumores malignos.

In [ ]:
# Separar características y variable objetivo
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"].map({"M": 1, "B": 0})

# Comprobación: no debe quedar ningún valor sin codificar
assert y.isnull().sum() == 0, "Hay valores de diagnosis distintos de 'M' y 'B'."

# Nombres de las clases en el orden de su código (0 = Benigno, 1 = Maligno)
NOMBRES_CLASES = ["Benigno", "Maligno"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("\nCodificación de y (0 = Benigno, 1 = Maligno):")
print(y.value_counts().sort_index().rename(index={0: "0 (Benigno)", 1: "1 (Maligno)"}))

### 2.2 División en entrenamiento y prueba
Se reserva el **20 %** de los datos para prueba (`test_size=0.2`) con `random_state=42` y
`stratify=y`, para que entrenamiento y prueba mantengan la misma proporción de benignos y malignos.

**Esta misma división la usan los tres algoritmos**, así la comparación es justa: todos aprenden
con los mismos ejemplos y se evalúan con los mismos tumores que nunca vieron.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba       : {X_test.shape[0]} registros\n")

# Verificación de la estratificación (proporción de malignos en cada conjunto)
proporciones = pd.DataFrame({
    "Total": y.value_counts(normalize=True),
    "Entrenamiento": y_train.value_counts(normalize=True),
    "Prueba": y_test.value_counts(normalize=True),
}).rename(index={0: "Benigno (0)", 1: "Maligno (1)"})
display((proporciones * 100).round(2).astype(str) + " %")

### 2.3 Estandarización con `StandardScaler`
La estandarización transforma cada variable para que tenga **media 0 y desviación estándar 1**:
$z = (x - \mu) / \sigma$.

**¿Por qué `fit` solo con entrenamiento?** La media ($\mu$) y la desviación ($\sigma$) se calculan
**únicamente con los datos de entrenamiento** (`fit_transform`) y luego se aplican tal cual a los
datos de prueba (`transform`). Si el escalador se ajustara con todo el dataset, información del
conjunto de prueba (sus medias y varianzas) se "filtraría" al entrenamiento (*data leakage*) y las
métricas serían optimistas: el modelo estaría usando, indirectamente, datos que se supone que no ha
visto. Así simulamos la situación real, en la que los pacientes nuevos llegan después de entrenar.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # aprende media y desviación SOLO del entrenamiento
X_test_scaled = scaler.transform(X_test)         # aplica esa misma transformación a la prueba

# Verificación: el entrenamiento queda con media ~0 y desviación ~1.
# La prueba queda cerca de esos valores, pero no exactos (es lo esperado: no se usó para ajustar).
print(f"Entrenamiento -> media: {X_train_scaled.mean():.4f} | desviación: {X_train_scaled.std():.4f}")
print(f"Prueba        -> media: {X_test_scaled.mean():.4f} | desviación: {X_test_scaled.std():.4f}")
print("\nForma de X_train_scaled:", X_train_scaled.shape)
print("Forma de X_test_scaled :", X_test_scaled.shape)

### 2.4 Variables compartidas y funciones de evaluación comunes
Para que los tres modelos se evalúen **exactamente igual**, se definen dos funciones que todos los
integrantes deben usar:

- `evaluar_modelo(nombre, y_test, y_pred)`: imprime accuracy, precision, recall, F1, el
  `classification_report`, dibuja la matriz de confusión y **devuelve un diccionario** con las
  métricas (sirve para armar la tabla comparativa).
- `graficar_matriz_confusion(cm, titulo, ax)`: dibuja una matriz de confusión como *heatmap* con
  etiquetas Benigno/Maligno (útil para poner las tres matrices lado a lado en la comparación).

| Variable | Contenido | Quién la usa |
|---|---|---|
| `X_train_scaled`, `X_test_scaled` | Características estandarizadas (entrenamiento / prueba) | KNN, SVM, MLP |
| `y_train`, `y_test` | Objetivo codificado (0 = Benigno, 1 = Maligno) | KNN, SVM, MLP |
| `RANDOM_STATE` | Semilla = 42 | Todos |
| `NOMBRES_CLASES` | `["Benigno", "Maligno"]` | Gráficas y reportes |
| `y_pred_knn`, `y_pred_svm`, `y_pred_mlp` | Predicciones finales sobre `X_test_scaled` | Comparación |
| `metricas_knn`, `metricas_svm`, `metricas_mlp` | Diccionarios devueltos por `evaluar_modelo` | Comparación |

In [ ]:
# ============================================================
# 2.4 Funciones de evaluación comunes (las usan los TRES modelos)
# ============================================================
def graficar_matriz_confusion(cm, titulo="Matriz de confusión", ax=None, cmap="Blues"):
    # Dibuja una matriz de confusión como heatmap con etiquetas Benigno/Maligno.
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, cbar=False,
                xticklabels=NOMBRES_CLASES, yticklabels=NOMBRES_CLASES,
                annot_kws={"size": 14}, linewidths=0.5, linecolor="white", ax=ax)
    ax.set_xlabel("Predicción del modelo")
    ax.set_ylabel("Diagnóstico real")
    ax.set_title(titulo)
    return ax


def evaluar_modelo(nombre, y_real, y_pred, mostrar=True):
    # Calcula las métricas (clase positiva = Maligno = 1), muestra el reporte y la
    # matriz de confusión, y devuelve un diccionario para la tabla comparativa.
    cm = confusion_matrix(y_real, y_pred, labels=[0, 1])
    vn, fp, fn, vp = cm.ravel()   # verdaderos neg., falsos pos., falsos neg., verdaderos pos.

    metricas = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_real, y_pred),
        "Precision": precision_score(y_real, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_real, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_real, y_pred, pos_label=1, zero_division=0),
        "VN": int(vn), "FP": int(fp), "FN": int(fn), "VP": int(vp),
        "Errores": int(fp + fn),
    }

    if mostrar:
        print(f"===== {nombre}: resultados en el conjunto de prueba =====")
        for clave in ["Accuracy", "Precision", "Recall", "F1-score"]:
            print(f"{clave:<10}: {metricas[clave]:.4f}")
        print(f"Errores   : {fp + fn} de {len(y_real)}  "
              f"(falsos positivos = {fp}, falsos negativos = {fn})\n")
        print("Classification report:")
        print(classification_report(y_real, y_pred, labels=[0, 1],
                                    target_names=NOMBRES_CLASES, digits=4))
        graficar_matriz_confusion(cm, f"Matriz de confusión — {nombre}")
        plt.tight_layout()
        plt.show()

    return metricas


print("Funciones evaluar_modelo() y graficar_matriz_confusion() listas para usarse.")

## 3. K-Nearest Neighbors (KNN)
> **Responsable:** _Nombre del integrante_

In [ ]:
# ============================================================
# SECCIÓN 3 — KNN   (a cargo de: ____________________ )
# ============================================================
# Usar ÚNICAMENTE las variables ya preparadas:
#     X_train_scaled, X_test_scaled, y_train, y_test   (y RANDOM_STATE si se necesita)
# NO volver a dividir ni a escalar los datos.
#
# Al terminar deben existir:
#     y_pred_knn   -> predicciones finales del modelo sobre X_test_scaled
#     metricas_knn = evaluar_modelo("KNN", y_test, y_pred_knn)

## 4. Support Vector Machine (SVM)
> **Responsable:** _Nombre del integrante_

In [ ]:
# ============================================================
# SECCIÓN 4 — SVM   (a cargo de: ____________________ )
# ============================================================
# Usar ÚNICAMENTE las variables ya preparadas:
#     X_train_scaled, X_test_scaled, y_train, y_test   (y RANDOM_STATE si se necesita)
# NO volver a dividir ni a escalar los datos.
#
# Al terminar deben existir:
#     y_pred_svm   -> predicciones finales del modelo sobre X_test_scaled
#     metricas_svm = evaluar_modelo("SVM", y_test, y_pred_svm)

## 5. Red neuronal artificial: Perceptrón Multicapa (MLP)
> **Responsable:** _Nombre del líder_

Un **MLP** es una red neuronal formada por una capa de entrada (30 neuronas, una por característica),
una o más **capas ocultas** y una capa de salida que estima la probabilidad de que el tumor sea maligno.
Cada neurona combina sus entradas con pesos, aplica una función de activación (aquí **ReLU**) y pasa
el resultado a la siguiente capa. Los pesos se ajustan con el optimizador **Adam**, que minimiza la
función de pérdida (*log-loss* o entropía cruzada) mediante retropropagación.

Plan de esta sección:
1. Entrenar un **modelo base** con los parámetros por defecto, como punto de referencia.
2. Buscar la mejor configuración con **GridSearchCV** (validación cruzada solo sobre el entrenamiento).
3. Entrenar el **modelo final**, revisar su **curva de pérdida** y evaluarlo en el conjunto de prueba.
4. Analizar los errores, en especial los **falsos negativos**.

### 5.1 Modelo base
Red con una capa oculta de 100 neuronas (valor por defecto de scikit-learn), activación ReLU y
solver Adam. Se usa `max_iter=1000` para dar iteraciones suficientes y que el entrenamiento converja.

In [ ]:
mlp_base = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
mlp_base.fit(X_train_scaled, y_train)
y_pred_mlp_base = mlp_base.predict(X_test_scaled)

print(f"Iteraciones (épocas) hasta converger: {mlp_base.n_iter_}")
print(f"Pérdida final de entrenamiento      : {mlp_base.loss_:.4f}")
print(f"Accuracy en prueba (referencia)     : {accuracy_score(y_test, y_pred_mlp_base):.4f}")
print(f"Recall maligno en prueba (referencia): {recall_score(y_test, y_pred_mlp_base):.4f}")

### 5.2 Búsqueda de hiperparámetros con GridSearchCV
Se prueban varias arquitecturas y valores de regularización:

| Hiperparámetro | Valores probados | Qué controla |
|---|---|---|
| `hidden_layer_sizes` | `(16,)`, `(32,)`, `(32, 16)`, `(64, 32)` | Número de capas ocultas y neuronas por capa |
| `alpha` | `0.0001`, `0.001`, `0.01` | Regularización L2: penaliza pesos grandes para reducir el sobreajuste |
| `learning_rate_init` | `0.001`, `0.01` | Tamaño del paso inicial de Adam |

Detalles importantes:
- **Validación cruzada estratificada de 5 pliegues** (`StratifiedKFold`) **solo con el entrenamiento**.
  El conjunto de prueba no se toca hasta la evaluación final, así la elección del modelo no se
  "contamina" con los datos de prueba.
- El escalador va **dentro de un `Pipeline`**: en cada pliegue se ajusta solo con la parte de
  entrenamiento de ese pliegue, evitando *data leakage* también durante la validación cruzada.
- Métrica de selección: **F1-score de la clase maligna**, porque equilibra *recall* (no dejar pasar
  tumores malignos) y *precision* (no generar demasiadas falsas alarmas). También se registran
  accuracy y recall.
- `early_stopping` se deja desactivado para conservar la curva de pérdida completa; para activarlo
  bastaría con poner `early_stopping=True` (usa el 10 % del entrenamiento como validación interna).

In [ ]:
# Pipeline: estandarización + red neuronal (los datos SIN escalar entran al pipeline)
pipeline_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(activation="relu", solver="adam", max_iter=2000,
                          early_stopping=False, random_state=RANDOM_STATE)),
])

param_grid = {
    "mlp__hidden_layer_sizes": [(16,), (32,), (32, 16), (64, 32)],
    "mlp__alpha": [1e-4, 1e-3, 1e-2],
    "mlp__learning_rate_init": [0.001, 0.01],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_mlp = GridSearchCV(
    estimator=pipeline_mlp,
    param_grid=param_grid,
    scoring={"accuracy": "accuracy", "recall": "recall", "f1": "f1"},
    refit="f1",          # el mejor modelo se elige por F1 de la clase maligna
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

n_combinaciones = int(np.prod([len(v) for v in param_grid.values()]))
print(f"Probando {n_combinaciones} combinaciones x 5 pliegues = {n_combinaciones * 5} entrenamientos...")
grid_mlp.fit(X_train, y_train)   # X_train sin escalar: el pipeline escala dentro de cada pliegue

# Mejores hiperparámetros (se quita el prefijo "mlp__" del pipeline)
mejores_parametros = {k.replace("mlp__", ""): v for k, v in grid_mlp.best_params_.items()}
print("\nMejores hiperparámetros:", mejores_parametros)
print(f"F1 medio en validación cruzada: {grid_mlp.best_score_:.4f}")

In [ ]:
# Tabla con las 10 mejores combinaciones según F1 en validación cruzada
resultados_cv = pd.DataFrame(grid_mlp.cv_results_)
resultados_cv["arquitectura"] = resultados_cv["param_mlp__hidden_layer_sizes"].astype(str)

tabla_cv = (resultados_cv
            .sort_values("mean_test_f1", ascending=False)
            [["arquitectura", "param_mlp__alpha", "param_mlp__learning_rate_init",
              "mean_test_accuracy", "mean_test_recall", "mean_test_f1", "std_test_f1",
              "mean_train_f1"]]
            .rename(columns={"param_mlp__alpha": "alpha",
                             "param_mlp__learning_rate_init": "learning_rate_init",
                             "mean_test_accuracy": "Accuracy CV",
                             "mean_test_recall": "Recall CV",
                             "mean_test_f1": "F1 CV",
                             "std_test_f1": "Desv. F1 CV",
                             "mean_train_f1": "F1 entrenamiento"})
            .head(10)
            .reset_index(drop=True))
display(tabla_cv.round(4))

### 5.3 Comparación de arquitecturas
Para cada arquitectura se toma su mejor combinación de `alpha` y `learning_rate_init` y se comparan
el F1 y el recall medios en validación cruzada (las barras de error indican la desviación estándar
entre los 5 pliegues). Si la diferencia entre arquitecturas es menor que la desviación, significa que
todas rinden de forma parecida en este problema.

In [ ]:
mejor_por_arquitectura = (resultados_cv
                          .sort_values("mean_test_f1", ascending=False)
                          .groupby("arquitectura", sort=False).head(1)
                          .set_index("arquitectura"))
orden = ["(16,)", "(32,)", "(32, 16)", "(64, 32)"]
mejor_por_arquitectura = mejor_por_arquitectura.loc[[a for a in orden if a in mejor_por_arquitectura.index]]

display(mejor_por_arquitectura[["param_mlp__alpha", "param_mlp__learning_rate_init",
                                "mean_test_accuracy", "mean_test_recall", "mean_test_f1"]]
        .rename(columns={"param_mlp__alpha": "alpha", "param_mlp__learning_rate_init": "learning_rate_init",
                         "mean_test_accuracy": "Accuracy CV", "mean_test_recall": "Recall CV",
                         "mean_test_f1": "F1 CV"}).round(4))

posiciones = np.arange(len(mejor_por_arquitectura))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(posiciones - ancho / 2, mejor_por_arquitectura["mean_test_f1"], ancho,
       yerr=mejor_por_arquitectura["std_test_f1"], capsize=4, label="F1 (CV)", color="#3A6EA5")
ax.bar(posiciones + ancho / 2, mejor_por_arquitectura["mean_test_recall"], ancho,
       yerr=mejor_por_arquitectura["std_test_recall"], capsize=4, label="Recall (CV)", color="#E09F3E")
ax.set_xticks(posiciones)
ax.set_xticklabels(mejor_por_arquitectura.index)
ax.set_xlabel("Neuronas por capa oculta (hidden_layer_sizes)")
ax.set_ylabel("Puntuación media en validación cruzada")
ax.set_ylim(0.85, 1.02)
ax.set_title("MLP: rendimiento por arquitectura (validación cruzada, 5 pliegues)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

### 5.4 Entrenamiento del modelo final
Se entrena la red con los **mejores hiperparámetros** encontrados, usando todo el conjunto de
entrenamiento ya escalado (`X_train_scaled`, el mismo que usan KNN y SVM). Como el escalador global
también se ajustó solo con el entrenamiento, este modelo es equivalente al mejor modelo del
`GridSearchCV`. Las predicciones finales se guardan en **`y_pred_mlp`**.

In [ ]:
mlp_final = MLPClassifier(
    activation="relu",
    solver="adam",
    max_iter=2000,
    early_stopping=False,
    random_state=RANDOM_STATE,
    **mejores_parametros,
)
mlp_final.fit(X_train_scaled, y_train)

# Predicciones finales sobre el conjunto de prueba (variable que usa la sección de comparación)
y_pred_mlp = mlp_final.predict(X_test_scaled)
# Probabilidad estimada de que cada tumor sea maligno (útil para el análisis de umbral)
y_proba_mlp = mlp_final.predict_proba(X_test_scaled)[:, 1]

# Comprobación de que coincide con el mejor modelo del GridSearchCV
coincide = np.array_equal(y_pred_mlp, grid_mlp.predict(X_test))
print("Configuración final:", mejores_parametros)
print(f"Épocas de entrenamiento: {mlp_final.n_iter_} | pérdida final: {mlp_final.loss_:.4f}")
print(f"¿Mismas predicciones que el mejor modelo del GridSearchCV?: {coincide}")

### 5.5 Curva de pérdida
La curva muestra cómo disminuye la función de pérdida (*log-loss*) en el entrenamiento a lo largo de
las épocas. Una curva que baja y se **estabiliza** indica que el modelo **convergió**. Se incluye
también la del modelo base como referencia.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(mlp_final.loss_curve_, label=f"MLP final {mejores_parametros['hidden_layer_sizes']}",
        color="#3A6EA5", linewidth=2)
ax.plot(mlp_base.loss_curve_, label="MLP base (100,)", color="#999999", linestyle="--")
ax.set_xlabel("Época (iteración)")
ax.set_ylabel("Pérdida de entrenamiento (log-loss)")
ax.set_title("Curva de pérdida del MLP")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()

### 5.6 Evaluación del MLP en el conjunto de prueba
Se usan las mismas métricas y la misma función de evaluación que en KNN y SVM.

In [ ]:
metricas_mlp = evaluar_modelo("MLP", y_test, y_pred_mlp)

### 5.7 Análisis de errores: falsos negativos y falsos positivos
En diagnóstico de cáncer los dos tipos de error **no cuestan lo mismo**:
- **Falso negativo (FN):** un tumor **maligno** clasificado como **benigno**. Es el error más grave,
  porque el paciente podría no recibir tratamiento a tiempo.
- **Falso positivo (FP):** un tumor **benigno** clasificado como **maligno**. Genera preocupación y
  estudios adicionales (por ejemplo, una biopsia), pero normalmente se corrige con pruebas posteriores.

Por eso, además del accuracy, el **recall de la clase maligna** es la métrica clave.

In [ ]:
fn = metricas_mlp["FN"]
fp = metricas_mlp["FP"]
total_malignos = int((y_test == 1).sum())

print(f"Tumores malignos en prueba     : {total_malignos}")
print(f"Malignos detectados (VP)       : {metricas_mlp['VP']}")
print(f"Falsos negativos (M -> B)      : {fn}")
print(f"Falsos positivos (B -> M)      : {fp}")
print(f"Recall maligno                 : {metricas_mlp['Recall']:.2%}")

# Detalle de los falsos negativos: qué probabilidad de malignidad les asignó la red
idx_fn = np.where((y_test.values == 1) & (y_pred_mlp == 0))[0]
if len(idx_fn) > 0:
    print("\nDetalle de los falsos negativos:")
    display(pd.DataFrame({
        "Índice en el dataset": y_test.index[idx_fn],
        "Prob. estimada de maligno": y_proba_mlp[idx_fn].round(4),
    }))
else:
    print("\nEl MLP no tuvo falsos negativos en el conjunto de prueba.")

**Análisis de umbral (ilustrativo).** Por defecto el MLP clasifica como maligno si la probabilidad
estimada es ≥ 0.5. Bajar ese umbral hace al modelo más "precavido": detecta más malignos (sube el
recall, bajan los FN) a cambio de más falsas alarmas (sube FP). La tabla siguiente muestra ese
compromiso. *Es solo un análisis exploratorio: el modelo final se mantiene con el umbral 0.5, ya que
elegir el umbral con los datos de prueba sería otra forma de* data leakage.

In [ ]:
filas = []
for umbral in [0.5, 0.4, 0.3, 0.2, 0.1]:
    y_umbral = (y_proba_mlp >= umbral).astype(int)
    vn_u, fp_u, fn_u, vp_u = confusion_matrix(y_test, y_umbral, labels=[0, 1]).ravel()
    filas.append({
        "Umbral": umbral,
        "Accuracy": accuracy_score(y_test, y_umbral),
        "Precision": precision_score(y_test, y_umbral, zero_division=0),
        "Recall": recall_score(y_test, y_umbral, zero_division=0),
        "FN (M -> B)": int(fn_u),
        "FP (B -> M)": int(fp_u),
    })
display(pd.DataFrame(filas).set_index("Umbral").round(4))

In [ ]:
# Interpretación automática de los resultados del MLP
print("INTERPRETACIÓN DE LOS RESULTADOS DEL MLP")
print("-" * 60)
print(f"- El MLP clasificó correctamente {metricas_mlp['VN'] + metricas_mlp['VP']} de {len(y_test)} "
      f"tumores (accuracy = {metricas_mlp['Accuracy']:.2%}).")
print(f"- De cada 100 tumores que el modelo marca como malignos, ~{metricas_mlp['Precision'] * 100:.0f} "
      f"realmente lo son (precision = {metricas_mlp['Precision']:.2%}).")
print(f"- Detectó {metricas_mlp['VP']} de {total_malignos} tumores malignos "
      f"(recall = {metricas_mlp['Recall']:.2%}).")
if fn == 0:
    print("- No hubo falsos negativos: ningún tumor maligno se clasificó como benigno.")
else:
    print(f"- Hubo {fn} falso(s) negativo(s): tumor(es) maligno(s) clasificado(s) como benigno(s). "
          "Es el error más costoso en este problema.")
print(f"- Hubo {fp} falso(s) positivo(s): tumor(es) benigno(s) marcado(s) como maligno(s).")

**Comentario de los resultados** *(valores obtenidos con `random_state=42`; si al reejecutar
cambian, actualizar este texto con los de las celdas anteriores)*.

- **Selección del modelo.** La mejor configuración fue una sola capa oculta de **32 neuronas**
  (`alpha=0.0001`, `learning_rate_init=0.01`), con F1 ≈ 0.973 en validación cruzada. Las cuatro
  arquitecturas quedaron dentro de una desviación estándar entre sí: este problema **no necesita una
  red profunda**; una red pequeña ya separa bien las clases. El F1 de entrenamiento = 1.0 frente a
  ≈ 0.97 en validación indica un ligero sobreajuste, normal en un dataset pequeño.
- **Convergencia.** La curva de pérdida baja rápidamente y se estabiliza (≈ 116 épocas), por lo que
  el entrenamiento convergió sin necesidad de más iteraciones.
- **Rendimiento en prueba.** Accuracy ≈ **97.4 %** (111 de 114 aciertos), precision = **100 %**
  (ningún benigno se marcó como maligno) y recall ≈ **92.9 %** (39 de 42 malignos detectados).
- **Falsos negativos.** Los 3 errores del modelo son todos **falsos negativos**: tumores malignos
  clasificados como benignos, que es justamente el error más grave. Dos de ellos reciben una
  probabilidad de malignidad casi nula, es decir, el modelo los "ve" como claramente benignos
  (probablemente son malignos con medidas atípicas, parecidas a las de un tumor benigno). Por eso
  bajar el umbral solo recupera uno de ellos (con umbral 0.2, FN pasa de 3 a 2 a cambio de 1 FP).
- **Conclusión para la comparación.** El MLP es muy preciso, pero su punto débil es el recall de la
  clase maligna. Al comparar con KNN y SVM conviene fijarse en **cuál tiene menos falsos negativos**,
  no solo en cuál tiene mayor accuracy.

## 6. Comparación de modelos
> **Responsable:** _Nombre del integrante_

In [ ]:
# ============================================================
# SECCIÓN 6 — Comparación de modelos   (a cargo de: ____________________ )
# ============================================================
# Variables disponibles (todas calculadas sobre los MISMOS datos de prueba, y_test):
#     y_pred_knn, y_pred_svm, y_pred_mlp            -> predicciones de cada modelo
#     metricas_knn, metricas_svm, metricas_mlp      -> diccionarios devueltos por evaluar_modelo()
#                                                      (Accuracy, Precision, Recall, F1-score, VN, FP, FN, VP, Errores)
#     y_proba_mlp                                   -> probabilidades del MLP (opcional, p. ej. curva ROC)
#     graficar_matriz_confusion(cm, titulo, ax)     -> para mostrar las 3 matrices lado a lado
#
# Debe incluir:
#     1) Tabla comparativa con las métricas de los tres algoritmos
#        (por ejemplo: pd.DataFrame([metricas_knn, metricas_svm, metricas_mlp]).set_index("Modelo"))
#     2) Al menos una gráfica que muestre las diferencias de rendimiento

## 7. Conclusiones
> **Responsable:** _Nombre del integrante_

In [ ]:
# ============================================================
# SECCIÓN 7 — Conclusiones   (a cargo de: Frida Natalia Dzul Chi )
# ============================================================
# Responder en celdas markdown, CON BASE EN LOS RESULTADOS de la sección 6
# (usar metricas_knn, metricas_svm, metricas_mlp y la tabla comparativa):
#     1) ¿Cuál de los tres algoritmos obtuvo el mejor rendimiento?
#     2) ¿Cuál presentó más errores de clasificación?   (ver la clave "Errores", "FN" y "FP")
#     3) ¿Qué diferencias observaste en su comportamiento?
#     4) ¿Elegirías el algoritmo con mayor exactitud para este problema? Justificar considerando
#        el tipo de clasificación (diagnóstico médico: costo de los falsos negativos -> recall).

---
### Verificación de integración (antes de entregar)
Esta celda revisa que todas las secciones hayan dejado sus variables y que las predicciones tengan
el tamaño correcto. Ejecutarla después de **Entorno de ejecución → Reiniciar y ejecutar todo**.

In [ ]:
print("VERIFICACIÓN DE INTEGRACIÓN")
print("-" * 50)
for nombre_var in ["y_pred_knn", "y_pred_svm", "y_pred_mlp",
                   "metricas_knn", "metricas_svm", "metricas_mlp"]:
    if nombre_var not in globals():
        print(f"[PENDIENTE] {nombre_var} todavía no existe")
    elif nombre_var.startswith("y_pred") and len(globals()[nombre_var]) != len(y_test):
        print(f"[ERROR]     {nombre_var} tiene {len(globals()[nombre_var])} predicciones; "
              f"se esperaban {len(y_test)}")
    else:
        print(f"[OK]        {nombre_var}")